In [ ]:
!python -m pip install --user datasets

In [1]:
import pandas as pd
from datasets import load_dataset


C:\Users\mansh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# !pip install -U g4f
# !pip install aiohttp
!python -m pip install --user -U g4f
!python -m pip install --user aiohttp


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import time
from g4f.client import Client

# Initialize the client
client = Client()

# List of input messages
messages = [
    "Tell me a joke on ML engineer"
]

total_time = 0
retry_limit = 3  # Maximum retries for a failed request

# Loop through each message, send it, and measure response time
for i, msg in enumerate(messages):
    print(f"Processing message {i + 1} of {len(messages)}...")

    for attempt in range(retry_limit):
        try:
            start_time = time.time()  # Start the timer

            # API request
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": msg}],
                web_search=False
            )

            end_time = time.time()  # End the timer
            elapsed_time = end_time - start_time
            total_time += elapsed_time

            print(f"Message {i + 1}: {msg}")
            print(f"Response: {response.choices[0].message.content}")
            print(f"Time taken: {elapsed_time:.2f} seconds\n")
            break  # Exit retry loop on success

        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt + 1 == retry_limit:
                print(f"Skipping message {i + 1} after {retry_limit} attempts.\n")
            else:
                print("Retrying...\n")
                time.sleep(1)  # Add delay before retrying

# Summary
print(f"Total time for {len(messages)} messages: {total_time:.2f} seconds")
if messages:
    print(f"Average time per message: {total_time / len(messages):.2f} seconds")


Processing message 1 of 1...
Message 1: Tell me a joke on ML engineer
Response: Why did the machine learning engineer break up with their partner?

Because they had too many "overfitting" issues!
Time taken: 5.47 seconds

Total time for 1 messages: 5.47 seconds
Average time per message: 5.47 seconds


In [4]:
#BLIP model
# BLIP model
import asyncio
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process stories from a DataFrame
async def main():
    # Load the DataFrame (Modify the path to your CSV file if needed)
    df = pd.read_csv("C:\\Users\\mansh\\OneDrive\\Desktop\\NanoVLM inferencing\\BLIP-base_story_completions_long.csv")
  # Assuming CSV has 'Partial Story' and 'Completed Story' columns
    
    # Create an instance of the Client
    client = Client()
    executor = ThreadPoolExecutor(max_workers=20)
    
    tasks = []
    for idx, row in df.iterrows():
        s_no = idx + 1  # Generating sequential serial numbers
        partial_story = row["original_story"]
        completed_story = row["completed_story"]
        prompt = create_prompt(s_no, partial_story, completed_story)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial_story, completed_story, task))
    
    # Gather results
    results = await asyncio.gather(*(task[3] for task in tasks))
    
    # Store responses in DataFrame
    # df["Response"] = results
    # df.to_csv("story_evaluation_results.csv", index=False)
    
    print("\nPrinting the responses")
    for (s_no, partial, completed), response in zip([(task[0], task[1], task[2]) for task in tasks], results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


<frozen abc>:123: RuntimeWarning: coroutine 'async_generator_to_list' was never awaited


Generated response:  **Serial Number: 13**

**Grading:**

1. **Grammar: 8/10**  
   The completion is mostly grammatically correct, but the sentence structure is somewhat simplistic. The phrase "in the cake" feels a bit awkward and could be improved for clarity.

2. **Creativity: 6/10**
Generated response:  Serial Number: 17

**Partial Story:**
The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are

**Completed Story:**
the kitty is very funny. it stands in an empty bowl while munching from another one. the dishes are in

**Grading:**

1. **Grammar (Score: 4/10):** 
   - The response shows a lack of capitalization for sentence beginnings, indicating a developing understanding of grammar rules.

2. **Creativity (Score: 2/10):**
   - The completion lacks originality and imagination as it simply reiterates part of the prompt without adding new content.

3. **Consistency (Score: 7/10):**
   - The continuation is consistent in terms of maintainin

In [5]:
#git

import asyncio
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process stories from a DataFrame
async def main():
    # Load the DataFrame (Modify the path to your CSV file if needed)
    df = pd.read_csv("C:\\Users\\mansh\\OneDrive\\Desktop\\NanoVLM inferencing\\git-base_story_completions_long.csv")
  # Assuming CSV has 'Partial Story' and 'Completed Story' columns
    
    # Create an instance of the Client
    client = Client()
    executor = ThreadPoolExecutor(max_workers=20)
    
    tasks = []
    for idx, row in df.iterrows():
        s_no = idx + 1  # Generating sequential serial numbers
        partial_story = row["original_story"]
        completed_story = row["completed_story"]
        prompt = create_prompt(s_no, partial_story, completed_story)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial_story, completed_story, task))
    
    # Gather results
    results = await asyncio.gather(*(task[3] for task in tasks))
    
    # Store responses in DataFrame
    # df["Response"] = results
    # df.to_csv("story_evaluation_results.csv", index=False)
    
    print("\nPrinting the responses")
    for (s_no, partial, completed), response in zip([(task[0], task[1], task[2]) for task in tasks], results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  **Serial Number: 3**

**Grading:**

1. **Grammar: 10/10**  
   The completion maintains proper grammar and sentence structure, matching the beginning text.

2. **Creativity: 6/10**  
   The completion does not add any new elements or imaginative details to the story. It simply repeats the beginning text without expansion.

3. **Consistency: 10/10**  
   The completion is consistent with the beginning text, maintaining the same theme and subject matter.

4. **Meaningfulness: 8/10**  
   The completion is meaningful as it describes the treats on the table, but it lacks additional context or narrative development.

5. **Plot: 5/10**  
   There is no development of a plot; the text remains static without any progression or action.

**Total Score: 39/50**

**Estimated Age of the Student: B: 4-5**  
The completion reflects a basic understanding of language and structure typical of a child in this age group, but the lack of creativity and plot development suggests they ar

In [6]:
# kosmos-2
# for 5.65M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process a list of stories
async def main():
    # Define your list of tuples (serial number, partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white", "<start> In a dark basement, there is a white toilet sitting on some steps. Around it, there are broken boards and debris.<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like", "<start> In a shiny bathroom, the walls sparkle like magic! There’s a big tree next to a toilet, and plants are growing out of the toilet bowl.<pad>"),
    ("3", "There is a big table full of yummy", "<start> There is a big table full of yummy treats! There are cookies, cakes, and colorful candies all together. In addition to the treats, there are a couple of bottles on the table. One of the bottles is filled with jelly beans and the other one has a cake in it. The table is set up with a fork and a knife, ready for someone to enjoy the delicious treats.<pad>"),
    ("4", "Pink cakes and lollipops rest on white", "<start> Pink cakes and lollipops rest on white tables and pink plates. Cake slices have forks beside them. In the background, there's a jar of sprinkles and a vase of flowers.<pad>"),
    ("5", "The cake is so colorful with chocolate and", "<start> The cake is so colorful with chocolate and white frosting. It has cute polka dots and sprinkles on it. There are two spoons next to the cake, one on the left side and the other in the middle of the image.<pad>"),
    ("6", "In a funny bathroom, there are two shiny", "<start> In a funny bathroom, there are two shiny toilets side by side. The floor is like a big black and white checkerboard.<pad>"),
    ("7", "In a happy green bathroom, there are funny", "<start> In a happy green bathroom, there are funny monkeys on the curtain. A shiny sink and a little toilet can be seen in the bathroom.<pad>"),
    ("8", "The bathroom has a white toilet and a tall", "<start> The bathroom has a white toilet and a tall sink. There's also a shower, but the walls look like they haven't been painted yet.<pad>"),
    ("9", "In a shiny bathroom, there is a big", "<start> In a shiny bathroom, there is a big toilet bowl sitting on the floor. A tall stall has a shower attached to the wall above it. There is also a sink in the bathroom.<pad>"),
    ("10", "There's a man on a shiny, old", "<start> There's a man on a shiny, old motorcycle. He wears a long coat and sits very still. The picture is black and white.<pad>"),
    ("11", "There's a big building with a clock", "<start> There's a big building with a clock inside that's tall and pointy. The church has a tall tower that is covered in a white sheet.<pad>"),
    ("12", "The green bowl is on the table.", "<start> The green bowl is on the table. It is full of little trees called broccoli. Some broccoli is green and some are dark green.<pad>"),
    ("13", "There is a big, yummy cake on a", "<start> There is a big, yummy cake on a shiny silver plate. It has white frosting and blue sprinkles in the shape of the letter 'F' on it.<pad>"),
    ("14", "A big parade is happening! A police", "<start> A big parade is happening! A police motorcycle zooms by with a shiny car behind it. People are watching the parade from the side of the road.<pad>"),
    ("15", "A fluffy cat is on a table.", "<start> A fluffy cat is on a table. It is sitting next to a bowl of fruit. There are bananas, apples, and oranges in the bowl.<pad>"),
    ("16", "The orange kitty sits on the table beside", "<start> The orange kitty sits on the table beside a bright bowl. It lays on the table, looking cozy and content.<pad>"),
    ("17", "The kitty is very funny. It stands in", "<start> The kitty is very funny. It stands in an empty bowl while munching from another one. The dishes are filled with food, and the cat seems to be enjoying its meal.<pad>"),
    ("18", "The cat is eating its food. It's", "<start> The cat is eating its food. It's funny because it's standing inside a bowl while munching. The cat has black and white fur.<pad>"),
    ("19", "A young man is sitting in a small", "<start> A young man is sitting in a small room with a computer. He wears a cozy sweatshirt and looks at the camera. In front of him, there is a keyboard, a mouse, and a cell phone. There are other people in the room, but they are not visible.<pad>"),
    ("20", "The toilet has a big, round light above", "<start> The toilet has a big, round light above it. Sunlight comes in from a round window in the wall.<pad>"),
    ("21", "In a tiny bathroom, there is a white", "<start> In a tiny bathroom, there is a white toilet. The roof is slanted, and a bright window lets in natural light.<pad>"),
    ("22", "There are tiny green beads and nuts inside", "<start> There are tiny green beads and nuts inside bamboo pieces. I see scissors next to brussel sprouts and shiny beads. There is a cup on the right side of the image.<pad>"),
    ("23", "A man sits at his desk with a big", "<start> A man sits at his desk with a big, yummy hotdog. He's wearing a blue shirt and working on his computer.<pad>"),
    ("24", "The bowl has yummy fruit like apples,", "<start> The bowl has yummy fruit like apples, bananas, and oranges. It's sitting on the wooden table. The blue plate is placed in front of the cat. There is a potted plant in the background.<pad>"),
    ("25", "In a big parking lot, two cool", "<start> In a big parking lot, two cool motorbikes sit next to a shiny car. One bike is blue with yellow flames, and the other is black with orange flames.<pad>")
]

    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    data = {
            "Partial": [story[1] for story in tasks],
            "Complete": [story[2] for story in tasks]
        }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("kosmos_long_desc__results.csv")

    print("@" *100)
    print("\n printing the responses")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  Serial Number: 3

**Grammar**: 9/10
The grammar is very good overall. There are no major mistakes, but it feels slightly advanced for a 3-4 year-old.

**Creativity**: 8/10
The story is creative and includes a variety of treats along with an interesting detail of bottles containing jelly beans and a cake.

**Consistency**: 9/10
The completion is consistent with the beginning of the text. It elaborates on the idea of yummy treats in a logical way.

**Meaningfulness**: 8/10
The story makes sense overall and has a clear theme of enjoying delicious treats on a table.

**Plot**: 7/10
The plot is simple and straightforward, which is appropriate for the age group, but could be more engaging with a small event or action.

**Total Score**: 41/50

**Estimated Age Group**: D: 8-9.
While the story is advanced in its structure and vocabulary, the simplicity of the plot suggests it might be from a student in the 8-9 year age group.
Generated response:  Serial Number: 2

**Grading